# Diode and energy scans

Load one or several NeXus scans, plot the diode signal against any scanned axis (including photon energy or magnetic field), and save both HDF5 data and a PNG in `processed/diode_scans/`. Multi-dimensional field/diode trace arrays are flattened in acquisition order, preserving hysteresis sweeps.

In [ ]:
from pathlib import Path
from getpass import getuser
import sys

def find_project_root(start=None):
    folder = Path(start or Path.cwd()).resolve()
    for candidate in (folder, *folder.parents):
        if (candidate / "library").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing library/.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from library.scan_workflow import load_scan_channel, save_diode_scans, scan_channels
print("Project root:", ROOT)

## Configuration

In [ ]:
USER = getuser()
SCAN_IDS = [150]
RAW_FOLDER = ROOT / "SOLEIL_2507" / "data" / "20250723"
FILENAME = "scanx_{scan_id:04d}.nxs"

# Examples: "energy" for an energy scan; "field", "mtesla", or the
# facility-specific magnet channel for a diode/field scan; or a motor name.
X_CHANNEL = "energy"  # change to the field-channel alias for a field scan
Y_CHANNEL = "diode"
NORMALIZATION_CHANNEL = None  # e.g. "goldmesh" or "i0"
OUTPUT_FOLDER = ROOT / "processed" / "diode_scans"

## Inspect channels (optional)

If a channel name is unknown, inspect the aliases discovered from the first file. Full HDF5 dataset paths are accepted too.

In [ ]:
first_file = RAW_FOLDER / FILENAME.format(scan_id=SCAN_IDS[0])
channels = scan_channels(first_file)
for alias, dataset_path in sorted(channels.items()):
    if "/" not in alias:
        print(f"{alias:30s} -> {dataset_path}")

## Load, plot, and save HDF5 + PNG

In [ ]:
scan_files = [RAW_FOLDER / FILENAME.format(scan_id=scan_id) for scan_id in SCAN_IDS]
xdata = [load_scan_channel(path, X_CHANNEL) for path in scan_files]
ydata = [load_scan_channel(path, Y_CHANNEL) for path in scan_files]
normalization = (
    None
    if NORMALIZATION_CHANNEL is None
    else [load_scan_channel(path, NORMALIZATION_CHANNEL) for path in scan_files]
)
h5_path, png_path = save_diode_scans(
    OUTPUT_FOLDER, SCAN_IDS, xdata, ydata,
    x_channel=X_CHANNEL, y_channel=Y_CHANNEL,
    normalization=normalization,
    normalization_channel=NORMALIZATION_CHANNEL, user=USER,
)
print("Saved HDF5:", h5_path)
print("Saved PNG:", png_path)